# RMOF-Net: Maize Nitrogen Classification

## 1. Introduction, Motivation, and Research Question

Nitrogen fertiliser management must balance crop productivity, cost, and environmental loss. Field RGB images are inexpensive to acquire, but nitrogen treatment is difficult to infer because plant colour, soil, shadows, genotype, and camera conditions vary together. This project studies three ordered treatment levels: **N0**, **N75**, and **NFull**.

**Research question.** Can a region-aware multi-scale model that combines deep visual features with leaf-colour evidence improve recognition of the ambiguous middle treatment ($N75$) while reducing severe $N0\leftrightarrow NFull$ errors?

The notebook contains the full reproducible workflow: data audit, preprocessing, baseline branches, RMOF-Net, evaluation utilities, final supplied results, and discussion. Training is disabled by default so that running all cells only audits data and renders the tables and figures embedded below.

## 2. Data Source and Task

The images come from Galic et al., *Nitrogen deficiency in maize: annotated image classification dataset*, Mendeley Data v1, DOI [10.17632/g7xnn2bm4g.1](https://doi.org/10.17632/g7xnn2bm4g.1) (CC BY 4.0). The extracted `Images/` directory contains RGB JPG images labelled by nitrogen treatment.

The task is three-class image classification with ordered labels $N0\mapsto0$, $N75\mapsto1$, and $NFull\mapsto2$. The active manifest is a deterministic, plot-disjoint 40/10/50 train/validation/test split. Keeping all images from one plot in one partition prevents plot and genotype leakage.

The next cell sets up the runtime and locates the experiment directory automatically.


In [ ]:
from __future__ import annotations

import argparse
import copy
import json
import math
import random
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from types import SimpleNamespace
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

CLASS_NAMES = ("N0", "N75", "NFull")
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

EXPERIMENT_ROOT = None
for _root in (Path.cwd(), *Path.cwd().parents):
    _candidate = _root / "EfficientN"
    if (_candidate / "split_40_10_50.csv").is_file() and (_candidate / "Images").is_dir():
        EXPERIMENT_ROOT = _candidate
        break
if EXPERIMENT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the bundled EfficientN dataset. Run this notebook from the repository or keep EfficientN/ beside it."
    )


## 3. Exploratory Data Analysis and Preprocessing

The data pipeline validates the manifest, preserves plot groups during optional training subsampling, applies ImageNet normalization, and uses geometric augmentation only. Colour jitter is deliberately excluded because leaf colour is part of the target signal.


In [ ]:
@dataclass
class ModelConfig:
    token_dim: int = 128
    dropout: float = 0.30
    use_stage3: bool = True
    use_stage4: bool = True
    use_region_tokens: bool = True
    use_color_stats: bool = True
    use_color_texture: bool = True
    fusion: str = "region_attention"  # concat, gate, attention, or colour_residual
    attention_heads: int = 4
    num_classes: int = 3


def preset_config(name: str) -> ModelConfig:
    """Return the requested main-ablation configuration."""
    base = ModelConfig()
    presets = {
        "cnn_baseline": dict(
            use_stage3=False,
            use_region_tokens=False,
            use_color_stats=False,
            use_color_texture=False,
            fusion="concat",
        ),
        "safe_deep_residual": dict(
            use_stage3=False,
            use_stage4=True,
            use_region_tokens=False,
            use_color_stats=False,
            use_color_texture=False,
            fusion="concat",
        ),
        "multiscale": dict(
            use_stage3=True,
            use_region_tokens=False,
            use_color_stats=False,
            use_color_texture=False,
            fusion="concat",
        ),
        "regions": dict(
            use_stage3=True,
            use_region_tokens=True,
            use_color_stats=False,
            use_color_texture=False,
            fusion="concat",
        ),
        "color_stats": dict(
            use_color_stats=True, use_color_texture=False, fusion="concat"
        ),
        "color_texture": dict(
            use_color_stats=True, use_color_texture=True, fusion="concat"
        ),
        "safe_colour_residual": dict(
            use_stage3=False,
            use_stage4=True,
            use_region_tokens=True,
            use_color_stats=True,
            use_color_texture=False,
            fusion="colour_residual",
        ),
        "region_cross_attention": dict(fusion="region_attention"),
        "ordinal_supervision": dict(fusion="region_attention"),
        "fusion_concat": dict(fusion="concat"),
        "fusion_gate": dict(fusion="gate"),
        "fusion_cross_attention": dict(fusion="cross_attention"),
        "fusion_region_attention": dict(fusion="region_attention"),
        "loss_ce": dict(fusion="region_attention"),
        "loss_ce_emd": dict(fusion="region_attention"),
        "loss_ce_score": dict(fusion="region_attention"),
        "loss_ce_emd_score": dict(fusion="region_attention"),
    }
    if name not in presets:
        raise ValueError(f"Unknown preset: {name}")
    return replace(base, **presets[name])


def loss_weights_for_preset(name: str) -> Tuple[float, float]:
    weights = {
        "ordinal_supervision": (0.25, 0.25),
        "loss_ce": (0.0, 0.0),
        "loss_ce_emd": (0.25, 0.0),
        "loss_ce_score": (0.0, 0.25),
        "loss_ce_emd_score": (0.25, 0.25),
    }
    return weights.get(name, (0.0, 0.0))


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True


In [ ]:
class MaizeDataset(Dataset):
    """One split from a CSV manifest of maize images."""

    def __init__(
        self,
        csv_path: Path,
        data_root: Path,
        split: str,
        transform: transforms.Compose,
        train_fraction: float = 1.0,
        seed: int = 42,
        cache_images: bool = False,
        image_size: int = 224,
    ) -> None:
        frame = pd.read_csv(csv_path)
        required = {"filepath", "filename", "label_index", "split"}
        missing = required.difference(frame.columns)
        if missing:
            raise ValueError(f"{csv_path} is missing columns: {sorted(missing)}")
        if not 0.0 < train_fraction <= 1.0:
            raise ValueError("train_fraction must be in (0, 1]")

        self.frame = frame.loc[frame["split"] == split].reset_index(drop=True)
        if split == "train" and train_fraction < 1.0:
            if "plot_id" in self.frame:
                plot_ids = np.sort(self.frame["plot_id"].unique())
                count = max(1, round(len(plot_ids) * train_fraction))
                selected = np.random.default_rng(seed).choice(
                    plot_ids, size=count, replace=False
                )
                self.frame = self.frame.loc[
                    self.frame["plot_id"].isin(selected)
                ].reset_index(drop=True)
            else:
                self.frame = self.frame.groupby(
                    "label_index", group_keys=False
                ).sample(frac=train_fraction, random_state=seed).reset_index(drop=True)
        if self.frame.empty:
            raise ValueError(f"No samples with split={split!r} in {csv_path}")

        self.data_root = data_root
        self.transform = transform
        self.cache_images = cache_images
        self.image_size = image_size
        self._image_cache: Dict[int, Image.Image] = {}

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, int, str]:
        row = self.frame.iloc[index]
        if self.cache_images and index in self._image_cache:
            image = self._image_cache[index].copy()
        else:
            image_path = self.data_root / str(row.filepath)
            if not image_path.is_file():
                raise FileNotFoundError(
                    f"Image not found: {image_path}. Set --data-root to the directory "
                    "that contains Images/."
                )
            with Image.open(image_path) as opened:
                image = opened.convert("RGB")
            if self.cache_images:
                image = image.resize(
                    (self.image_size, self.image_size), Image.Resampling.BILINEAR
                )
                self._image_cache[index] = image.copy()
        return self.transform(image), int(row.label_index), str(row.filename)


def build_transforms(
    image_size: int, augmentation: str, images_pre_resized: bool = False
) -> Tuple[transforms.Compose, transforms.Compose]:
    """Build ImageNet-normalized train and evaluation transforms."""
    normalise = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    train_steps: list[object] = [transforms.RandomHorizontalFlip()]
    if not images_pre_resized:
        train_steps.insert(0, transforms.Resize((image_size, image_size)))
    if augmentation == "strong":
        train_steps.extend(
            (transforms.RandomVerticalFlip(p=0.2), transforms.RandomRotation(15))
        )
    elif augmentation == "leaf":
        train_steps.extend(
            (
                transforms.RandomRotation(8),
                transforms.RandomAffine(
                    degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1)
                ),
            )
        )
    train_steps.extend((transforms.ToTensor(), normalise))

    eval_steps: list[object] = [transforms.ToTensor(), normalise]
    if not images_pre_resized:
        eval_steps.insert(0, transforms.Resize((image_size, image_size)))
    return transforms.Compose(train_steps), transforms.Compose(eval_steps)


def make_loaders(args: argparse.Namespace) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Create train, validation, and test loaders from the experiment arguments."""
    csv_path = Path(args.csv)
    if not csv_path.is_file():
        raise FileNotFoundError(f"Split file not found: {csv_path}")

    cache_images = getattr(args, "cache_images", False)
    train_transform, eval_transform = build_transforms(
        args.image_size, args.augmentation, images_pre_resized=cache_images
    )
    splits = set(pd.read_csv(csv_path)["split"].unique())
    validation_split = "val" if "val" in splits else "test"
    test_split = "test" if "test" in splits else validation_split
    root = Path(args.data_root)

    train_options: dict[str, object] = {
        "batch_size": args.batch_size,
        "num_workers": args.num_workers,
    }
    # Evaluation runs after every epoch, so another persistent worker pool
    # only oversubscribes the host without improving throughput.
    evaluation_options: dict[str, object] = {
        "batch_size": args.batch_size,
        "num_workers": 0,
    }
    if args.device.startswith("cuda"):
        train_options["pin_memory"] = True
        evaluation_options["pin_memory"] = True
    if args.num_workers > 0:
        train_options["persistent_workers"] = True

    train_generator = torch.Generator().manual_seed(args.seed)
    train = DataLoader(
        MaizeDataset(
            csv_path,
            root,
            "train",
            train_transform,
            args.train_fraction,
            args.seed,
            cache_images,
            args.image_size,
        ),
        shuffle=True,
        generator=train_generator,
        **train_options,
    )
    validation = DataLoader(
        MaizeDataset(
            csv_path,
            root,
            validation_split,
            eval_transform,
            cache_images=cache_images,
            image_size=args.image_size,
        ),
        **evaluation_options,
    )
    test = DataLoader(
        MaizeDataset(
            csv_path,
            root,
            test_split,
            eval_transform,
            cache_images=cache_images,
            image_size=args.image_size,
        ),
        **evaluation_options,
    )
    return train, validation, test


### 3.1 Data Audit

This audit reports the manifest schema, split and class counts, plot-group counts, image modes and dimensions, and one example per class. The first chart makes any class/split imbalance visible; the sample panel is a direct visual check of the RGB inputs.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

manifest_path = EXPERIMENT_ROOT / "split_40_10_50.csv"
manifest = pd.read_csv(manifest_path)
required_columns = {"filepath", "filename", "label", "label_index", "plot_id", "split"}
missing_columns = required_columns.difference(manifest.columns)
if missing_columns:
    raise ValueError(f"Manifest is missing columns: {sorted(missing_columns)}")

split_class_counts = pd.crosstab(manifest["split"], manifest["label"])
plot_counts = manifest.groupby("split")["plot_id"].nunique().rename("plots")

# Audit file dimensions without altering source images.
def image_properties(relative_path: str) -> tuple[str, tuple[int, int]]:
    with Image.open(EXPERIMENT_ROOT / relative_path) as image:
        return image.mode, image.size

properties = manifest["filepath"].map(image_properties)
manifest = manifest.assign(
    image_mode=properties.map(lambda value: value[0]),
    image_size=properties.map(lambda value: value[1]),
)

print(f"Images: {len(manifest)} | plots: {manifest['plot_id'].nunique()} | classes: {manifest['label'].nunique()}")
print("\nImages per split and class:")
display(split_class_counts)
print("\nDistinct plots per split:")
display(plot_counts.to_frame())
print("\nImage modes and dimensions:")
display(manifest.groupby(["image_mode", "image_size"]).size().rename("count").reset_index())

fig, axis = plt.subplots(figsize=(5.2, 3.5), constrained_layout=True)
split_class_counts.T.plot(kind="bar", ax=axis, color=["#4A90D9", "#5BA58B", "#D4A252"])
axis.set_title("Class balance by split")
axis.set_xlabel("Treatment")
axis.set_ylabel("Images")
axis.tick_params(axis="x", rotation=0)
axis.legend(title="Split")
plt.show()

# Use a separate compact figure so each class image is readable.
fig_samples, sample_axes = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)
for axis, label in zip(sample_axes, CLASS_NAMES):
    sample = manifest.loc[manifest["label"] == label].sort_values("filename").iloc[0]
    with Image.open(EXPERIMENT_ROOT / sample.filepath) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(label)
    axis.set_axis_off()
plt.show()


### 3.2 Preprocessing and Challenges

Images are resized to $224\times224$, converted to tensors, and normalized with ImageNet mean and standard deviation. Training uses horizontal flips and optional geometry-only rotations/affine transforms. The difficult aspects are: (1) avoiding plot-level leakage, (2) distinguishing the visually intermediate $N75$ treatment, (3) preserving colour evidence without allowing colour augmentation to erase it, and (4) handling backgrounds, shadows, weeds, and viewpoint variation.


## 4. Models and Methods

### 4.1 Existing Work and Group Contribution

The dataset is provided by Galic et al. The ResNet18 and EfficientNet-B0 backbones use the standard ImageNet-pretrained implementations from `torchvision`; DeiT-Tiny uses `timm`. These libraries supply baseline architectures only.

The project-specific work is the plot-disjoint manifest, the preprocessing policy, the RMOF auxiliary branch, foreground-weighted colour statistics, learned colour texture tokens, regional fusion, ordinal objectives, and the shared evaluation protocol. The baseline implementations are shown first for an explicit comparison point.

### 4.2 Baseline Branches


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 3) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.GELU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.GELU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.GELU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, images: torch.Tensor) -> Dict[str, Optional[torch.Tensor]]:
        return {"logits": self.classifier(self.encoder(images).flatten(1)), "score": None}


class ClassifierWrapper(nn.Module):
    def __init__(self, model: nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(self, images: torch.Tensor) -> Dict[str, Optional[torch.Tensor]]:
        return {"logits": self.model(images), "score": None}


def build_baseline_model(
    model_name: str, config: ModelConfig, pretrained: bool
) -> nn.Module:
    """Build one of the non-RMOF comparison models."""
    if model_name == "simple_cnn":
        return SimpleCNN(config.num_classes)
    if model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Sequential(
            nn.Dropout(p=config.dropout, inplace=True),
            nn.Linear(model.classifier[1].in_features, config.num_classes),
        )
        return ClassifierWrapper(model)
    if model_name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, config.num_classes)
        return ClassifierWrapper(model)
    if model_name == "deit_tiny":
        try:
            import timm
        except ImportError as error:
            raise ImportError("DeiT-Tiny requires `pip install timm`.") from error
        return ClassifierWrapper(
            timm.create_model("deit_tiny_patch16_224", pretrained=pretrained, num_classes=config.num_classes)
        )
    raise ValueError(f"Unknown baseline model: {model_name}")


def build_model(model_name: str, config: ModelConfig, pretrained: bool) -> nn.Module:
    """Dispatch the RMOF main branch or one isolated baseline branch."""
    if model_name == "rmof_efficientnet":
        return EfficientNetColourFusion(config, pretrained)
    return build_baseline_model(model_name, config, pretrained)


### 4.3 Core RMOF-Net Architecture

RMOF-Net keeps the pretrained EfficientNet classifier and adds a zero-initialized logit residual. It extracts global and $2\times2$ regional tokens from two deep stages, combines them with foreground-weighted colour statistics and a learned colour-texture encoder, then fuses the token streams with the selected mechanism.


In [ ]:
class SpatialTokenExtractor(nn.Module):
    """One global token and, optionally, four row-major 2x2 region tokens."""

    def __init__(self, channels: int, token_dim: int) -> None:
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(channels, token_dim), nn.GELU(), nn.LayerNorm(token_dim)
        )

    def forward(self, feature: torch.Tensor, regions: bool) -> torch.Tensor:
        pooled = F.adaptive_avg_pool2d(feature, 1).flatten(2)
        if regions:
            pooled = torch.cat((pooled, F.adaptive_avg_pool2d(feature, 2).flatten(2)), dim=2)
        tokens = pooled.transpose(1, 2)
        return self.project(tokens)


def rgb_to_hsv(rgb: torch.Tensor) -> torch.Tensor:
    """Differentiable RGB-to-HSV conversion for RGB values in [0, 1]."""
    maxc, max_index = rgb.max(dim=1)
    minc = rgb.min(dim=1).values
    delta = maxc - minc
    r, g, b = rgb[:, 0], rgb[:, 1], rgb[:, 2]
    safe_delta = delta.clamp_min(1e-6)
    hue = torch.zeros_like(maxc)
    hue = torch.where(max_index == 0, torch.remainder((g - b) / safe_delta, 6.0), hue)
    hue = torch.where(max_index == 1, (b - r) / safe_delta + 2.0, hue)
    hue = torch.where(max_index == 2, (r - g) / safe_delta + 4.0, hue)
    hue = torch.where(delta > 1e-6, hue / 6.0, torch.zeros_like(hue))
    saturation = torch.where(maxc > 1e-6, delta / maxc.clamp_min(1e-6), torch.zeros_like(maxc))
    return torch.stack((hue, saturation, maxc), dim=1)


def rgb_to_lab(rgb: torch.Tensor) -> torch.Tensor:
    """Approximate sRGB to CIE Lab conversion, sufficient for colour statistics."""
    linear_rgb = torch.where(
        rgb > 0.04045, ((rgb + 0.055) / 1.055).pow(2.4), rgb / 12.92
    )
    matrix = rgb.new_tensor(
        [[0.4124564, 0.3575761, 0.1804375], [0.2126729, 0.7151522, 0.0721750], [0.0193339, 0.1191920, 0.9503041]]
    )
    xyz = torch.einsum("ij,bjhw->bihw", matrix, linear_rgb)
    reference_white = rgb.new_tensor([0.95047, 1.0, 1.08883]).view(1, 3, 1, 1)
    xyz = xyz / reference_white
    delta = 6.0 / 29.0
    f_xyz = torch.where(
        xyz > delta**3, xyz.pow(1.0 / 3.0), xyz / (3 * delta**2) + 4.0 / 29.0
    )
    l = 116.0 * f_xyz[:, 1] - 16.0
    a = 500.0 * (f_xyz[:, 0] - f_xyz[:, 1])
    b = 200.0 * (f_xyz[:, 1] - f_xyz[:, 2])
    return torch.stack((l, a, b), dim=1)


class SoftForegroundMask(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.threshold = nn.Parameter(torch.tensor(-0.03))
        self.log_sharpness = nn.Parameter(torch.tensor(12.0))

    def forward(self, rgb: torch.Tensor) -> torch.Tensor:
        exg = 2.0 * rgb[:, 1:2] - rgb[:, 0:1] - rgb[:, 2:3]
        sharpness = F.softplus(self.log_sharpness)
        return torch.sigmoid(sharpness * (exg - self.threshold))


def weighted_region_statistics(
    features: torch.Tensor, mask: torch.Tensor, regions: bool
) -> torch.Tensor:
    """Return mask-weighted mean and standard deviation per global/2x2 region."""
    def pool(grid: int) -> torch.Tensor:
        denominator = F.adaptive_avg_pool2d(mask, grid).clamp_min(1e-5)
        mean = F.adaptive_avg_pool2d(features * mask, grid) / denominator
        second_moment = F.adaptive_avg_pool2d(features.square() * mask, grid) / denominator
        std = (second_moment - mean.square()).clamp_min(0.0).sqrt()
        return torch.cat((mean, std), dim=1).flatten(2).transpose(1, 2)

    global_stats = pool(1)
    return torch.cat((global_stats, pool(2)), dim=1) if regions else global_stats


class ColourBranch(nn.Module):
    """Soft leaf mask, explicit statistics and a shallow learned colour texture encoder."""

    def __init__(self, token_dim: int, use_stats: bool, use_texture: bool) -> None:
        super().__init__()
        self.use_stats = use_stats
        self.use_texture = use_texture
        self.mask = SoftForegroundMask()
        if use_stats:
            self.stats_project = nn.Sequential(
                nn.Linear(18, token_dim), nn.GELU(), nn.LayerNorm(token_dim)
            )
        if use_texture:
            texture_channels = max(32, token_dim // 2)
            self.texture = nn.Sequential(
                nn.Conv2d(3, texture_channels, 3, stride=2, padding=1),
                nn.BatchNorm2d(texture_channels),
                nn.GELU(),
                nn.Conv2d(texture_channels, token_dim, 3, stride=2, padding=1),
                nn.BatchNorm2d(token_dim),
                nn.GELU(),
            )
            self.texture_project = nn.Sequential(nn.LayerNorm(token_dim), nn.GELU())
        self.merge = (
            nn.Sequential(nn.Linear(2 * token_dim, token_dim), nn.GELU(), nn.LayerNorm(token_dim))
            if use_stats and use_texture
            else None
        )

    def forward(self, rgb: torch.Tensor, regions: bool) -> torch.Tensor:
        mask = self.mask(rgb)
        tokens: List[torch.Tensor] = []
        if self.use_stats:
            hsv = rgb_to_hsv(rgb)
            lab_scale = rgb.new_tensor((100.0, 128.0, 128.0)).view(1, 3, 1, 1)
            lab = rgb_to_lab(rgb) / lab_scale
            r, g, b = rgb[:, 0:1], rgb[:, 1:2], rgb[:, 2:3]
            exg = (2.0 * g - r - b) / 2.0
            exgr = (3.0 * g - 2.4 * r - b) / 3.4
            ngrdi = (g - r) / (g + r).clamp_min(1e-5)
            features = torch.cat((hsv, lab, exg, exgr, ngrdi), dim=1)
            tokens.append(self.stats_project(weighted_region_statistics(features, mask, regions)))
        if self.use_texture:
            texture = self.texture(rgb * mask)
            token_grid = 2 if regions else 1
            local = F.adaptive_avg_pool2d(texture, token_grid).flatten(2).transpose(1, 2)
            if regions:
                global_token = F.adaptive_avg_pool2d(texture, 1).flatten(2).transpose(1, 2)
                local = torch.cat((global_token, local), dim=1)
            tokens.append(self.texture_project(local))
        return tokens[0] if len(tokens) == 1 else self.merge(torch.cat(tokens, dim=-1))


class TokenAttentionPool(nn.Module):
    """Learn a token summary, starting from an unbiased uniform average."""

    def __init__(self, token_dim: int) -> None:
        super().__init__()
        self.norm = nn.LayerNorm(token_dim)
        self.score = nn.Linear(token_dim, 1)
        nn.init.zeros_(self.score.weight)
        nn.init.zeros_(self.score.bias)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        weights = self.score(self.norm(tokens)).softmax(dim=1)
        return (weights * tokens).sum(dim=1)


class CrossAttention(nn.Module):
    """Cross-attention with optional learnable region-to-region attention bias."""

    def __init__(self, token_dim: int, heads: int, aligned: bool) -> None:
        super().__init__()
        if token_dim % heads:
            raise ValueError("token_dim must be divisible by attention_heads")
        self.heads = heads
        self.head_dim = token_dim // heads
        self.aligned = aligned
        self.query = nn.Linear(token_dim, token_dim)
        self.key = nn.Linear(token_dim, token_dim)
        self.value = nn.Linear(token_dim, token_dim)
        self.out = nn.Linear(token_dim, token_dim)
        self.query_norm = nn.LayerNorm(token_dim)
        self.key_value_norm = nn.LayerNorm(token_dim)
        self.feed_forward_norm = nn.LayerNorm(token_dim)
        self.feed_forward = nn.Sequential(
            nn.Linear(token_dim, 2 * token_dim), nn.GELU(), nn.Linear(2 * token_dim, token_dim)
        )
        self.region_bias = nn.Parameter(torch.zeros(5, 5)) if aligned else None
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)
        nn.init.zeros_(self.feed_forward[-1].weight)
        nn.init.zeros_(self.feed_forward[-1].bias)

    def forward(
        self,
        query_tokens: torch.Tensor,
        key_value_tokens: torch.Tensor,
        query_regions: torch.Tensor,
        key_regions: torch.Tensor,
    ) -> torch.Tensor:
        batch, query_count, token_dim = query_tokens.shape
        key_count = key_value_tokens.shape[1]
        normalised_query = self.query_norm(query_tokens)
        normalised_key_value = self.key_value_norm(key_value_tokens)
        q = self.query(normalised_query).reshape(batch, query_count, self.heads, self.head_dim).transpose(1, 2)
        k = self.key(normalised_key_value).reshape(batch, key_count, self.heads, self.head_dim).transpose(1, 2)
        v = self.value(normalised_key_value).reshape(batch, key_count, self.heads, self.head_dim).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        if self.aligned:
            bias = self.region_bias[query_regions[:, None], key_regions[None, :]]
            scores = scores + bias.unsqueeze(0).unsqueeze(0)
        attention = scores.softmax(dim=-1)
        attended = (attention @ v).transpose(1, 2).reshape(batch, query_count, token_dim)
        fused = query_tokens + self.out(attended)
        return fused + self.feed_forward(self.feed_forward_norm(fused))


class EfficientNetColourFusion(nn.Module):
    """EfficientNet-B0 plus a safely initialized colour-and-region logit residual."""

    def __init__(self, config: ModelConfig, pretrained: bool = False) -> None:
        super().__init__()
        if not (config.use_stage3 or config.use_stage4):
            raise ValueError("At least one CNN stage must be enabled")
        self.config = config
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        backbone = models.efficientnet_b0(weights=weights)
        self.features = backbone.features
        self.avgpool = backbone.avgpool
        backbone.classifier[1] = nn.Sequential(
            nn.Dropout(p=config.dropout, inplace=True),
            nn.Linear(backbone.classifier[1].in_features, config.num_classes),
        )
        self.base_classifier = backbone.classifier
        self.stage3_tokens = SpatialTokenExtractor(112, config.token_dim)
        self.stage4_tokens = SpatialTokenExtractor(320, config.token_dim)
        self.deep_pool = TokenAttentionPool(config.token_dim)
        self.colour: Optional[ColourBranch]
        if config.use_color_stats or config.use_color_texture:
            self.colour = ColourBranch(
                config.token_dim, config.use_color_stats, config.use_color_texture
            )
        else:
            self.colour = None
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))
        if self.colour is not None and config.fusion == "gate":
            self.gate = nn.Sequential(nn.Linear(2 * config.token_dim, config.token_dim), nn.Sigmoid())
            nn.init.zeros_(self.gate[0].weight)
            nn.init.constant_(self.gate[0].bias, math.log(0.9 / 0.1))
        else:
            self.gate = None
        if self.colour is not None and config.fusion in {"cross_attention", "region_attention"}:
            self.cross_attention = CrossAttention(
                config.token_dim,
                config.attention_heads,
                aligned=config.fusion == "region_attention",
            )
        else:
            self.cross_attention = None
        self.colour_pool = TokenAttentionPool(config.token_dim) if self.colour is not None else None
        head_dim = 2 * config.token_dim if self.colour is not None and config.fusion == "concat" else config.token_dim
        self.aux_classifier = nn.Sequential(
            nn.LayerNorm(head_dim), nn.Dropout(config.dropout), nn.Linear(head_dim, config.num_classes)
        )
        nn.init.zeros_(self.aux_classifier[-1].weight)
        nn.init.zeros_(self.aux_classifier[-1].bias)
        self.score_head = nn.Sequential(nn.LayerNorm(head_dim), nn.Linear(head_dim, 1))
        self.base_frozen = False

    def freeze_base(self) -> None:
        """Freeze the validated EfficientNet path while fitting the logit residual."""
        for module in (self.features, self.base_classifier):
            module.requires_grad_(False)
            module.eval()
        if self.config.fusion == "colour_residual":
            # These deep-token modules are intentionally bypassed by the
            # low-capacity colour-only correction.
            for module in (self.stage3_tokens, self.stage4_tokens, self.deep_pool):
                module.requires_grad_(False)
                module.eval()
        self.base_frozen = True

    def _deep_tokens(
        self, images: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        stage3 = stage4 = None
        x = images
        for index, block in enumerate(self.features):
            x = block(x)
            if index == 5:
                stage3 = x  # 112 channels, 14x14 for 224x224 input
            elif index == 7:
                stage4 = x  # 320 channels, 7x7 for 224x224 input
        token_sets: List[torch.Tensor] = []
        region_ids: List[torch.Tensor] = []
        base_regions = torch.arange(5 if self.config.use_region_tokens else 1, device=images.device)
        if self.config.use_stage3:
            if stage3 is None:
                raise RuntimeError("EfficientNet stage 3 was not produced")
            token_sets.append(self.stage3_tokens(stage3, self.config.use_region_tokens))
            region_ids.append(base_regions)
        if self.config.use_stage4:
            if stage4 is None:
                raise RuntimeError("EfficientNet stage 4 was not produced")
            token_sets.append(self.stage4_tokens(stage4, self.config.use_region_tokens))
            region_ids.append(base_regions)
        return torch.cat(token_sets, dim=1), torch.cat(region_ids), x

    def forward(self, images: torch.Tensor) -> Dict[str, Optional[torch.Tensor]]:
        deep_tokens, deep_regions, final_features = self._deep_tokens(images)
        base_logits = self.base_classifier(self.avgpool(final_features).flatten(1))
        deep_summary = self.deep_pool(deep_tokens)
        if self.colour is None:
            fused = deep_summary
        else:
            rgb = (images * self.std + self.mean).clamp(0.0, 1.0)
            colour_tokens = self.colour(rgb, self.config.use_region_tokens)
            colour_summary = self.colour_pool(colour_tokens)
            colour_regions = torch.arange(colour_tokens.shape[1], device=images.device)
            if self.config.fusion == "colour_residual":
                fused = colour_summary
            elif self.config.fusion == "concat":
                fused = torch.cat((deep_summary, colour_summary), dim=-1)
            elif self.config.fusion == "gate":
                gate = self.gate(torch.cat((deep_summary, colour_summary), dim=-1))
                fused = gate * deep_summary + (1.0 - gate) * colour_summary
            elif self.config.fusion in {"cross_attention", "region_attention"}:
                fused_tokens = self.cross_attention(
                    deep_tokens, colour_tokens, deep_regions, colour_regions
                )
                fused = self.deep_pool(fused_tokens)
            else:
                raise ValueError(f"Unknown fusion mode: {self.config.fusion}")
        aux_logits = self.aux_classifier(fused)
        return {
            "logits": base_logits + aux_logits,
            "base_logits": base_logits,
            "aux_logits": aux_logits,
            "score": self.score_head(fused),
        }


### 4.4 Objective, Metrics, and Training Procedure

Cross-entropy is always present. Optional cumulative EMD and Smooth L1 score losses encode the ordered treatment structure. Model selection uses validation Macro-F1; evaluation reports class-aware, deficiency, ordinal, and efficiency metrics.


In [ ]:
def load_frozen_efficientnet_base(
    model: EfficientNetColourFusion, checkpoint_path: Path
) -> None:
    """Load a ClassifierWrapper EfficientNet checkpoint into the residual model."""
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Base checkpoint not found: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    source = checkpoint.get("state_dict", checkpoint)
    mapped: Dict[str, torch.Tensor] = {}
    for key, value in source.items():
        if key.startswith("model.features."):
            mapped["features." + key.removeprefix("model.features.")] = value
        elif key.startswith("model.classifier."):
            mapped["base_classifier." + key.removeprefix("model.classifier.")] = value
        elif key.startswith("features.") or key.startswith("base_classifier."):
            mapped[key] = value
    if not mapped:
        raise ValueError(f"No EfficientNet feature/classifier weights found in {checkpoint_path}")
    incompatible = model.load_state_dict(mapped, strict=False)
    unexpected = [key for key in incompatible.unexpected_keys if key in mapped]
    if unexpected:
        raise ValueError(f"Unexpected base checkpoint keys: {unexpected}")
    model.freeze_base()


class OrdinalLoss(nn.Module):
    def __init__(self, emd_weight: float, score_weight: float, label_smoothing: float = 0.0) -> None:
        super().__init__()
        self.cross_entropy = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.emd_weight = emd_weight
        self.score_weight = score_weight

    def forward(self, output: Dict[str, Optional[torch.Tensor]], target: torch.Tensor) -> Dict[str, torch.Tensor]:
        logits = output["logits"]
        ce = self.cross_entropy(logits, target)
        probabilities = logits.softmax(dim=1)
        one_hot = F.one_hot(target, num_classes=logits.shape[1]).float()
        emd = (probabilities.cumsum(dim=1) - one_hot.cumsum(dim=1)).square().mean()
        score = output["score"]
        if self.score_weight and score is None:
            raise ValueError("Score regression was selected but this baseline has no score head")
        score_loss = (
            F.smooth_l1_loss(score.squeeze(1), target.float()) if score is not None else ce.new_zeros(())
        )
        total = ce + self.emd_weight * emd + self.score_weight * score_loss
        return {"total": total, "ce": ce, "emd": emd, "score": score_loss}


In [ ]:
def metrics_from_predictions(
    labels: Sequence[int], predictions: Sequence[int]
) -> Dict[str, float]:
    """Compute the classification and ordinal metrics reported by the project."""
    labels_array = np.asarray(labels)
    predictions_array = np.asarray(predictions)
    f1s = f1_score(
        labels_array,
        predictions_array,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )
    deficiency_labels = labels_array != 2
    deficiency_predictions = predictions_array != 2
    matrix = confusion_matrix(labels_array, predictions_array, labels=[0, 1, 2])
    endpoint_total = int(matrix[0].sum() + matrix[2].sum())
    endpoint_errors = int(matrix[0, 2] + matrix[2, 0])
    return {
        "accuracy": float(accuracy_score(labels_array, predictions_array)),
        "macro_f1": float(
            f1_score(labels_array, predictions_array, average="macro", zero_division=0)
        ),
        "f1_deficiency": float(
            f1_score(deficiency_labels, deficiency_predictions, zero_division=0)
        ),
        "f1_n0": float(f1s[0]),
        "f1_n75": float(f1s[1]),
        "f1_nfull": float(f1s[2]),
        "ordinal_mae": float(np.abs(labels_array - predictions_array).mean()),
        "endpoint_confusion": endpoint_errors,
        "endpoint_confusion_rate": (
            float(endpoint_errors / endpoint_total) if endpoint_total else 0.0
        ),
    }


def save_confusion(matrix: np.ndarray, target: Path, title: str) -> None:
    """Save a consistently labelled three-class confusion matrix."""
    import matplotlib.pyplot as plt

    figure, axis = plt.subplots(figsize=(4.6, 4.0))
    image = axis.imshow(matrix, cmap="Blues")
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            axis.text(column, row, str(matrix[row, column]), ha="center", va="center")
    axis.set(
        xticks=range(3),
        yticks=range(3),
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
    )
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")
    axis.set_title(title)
    figure.colorbar(image, ax=axis, shrink=0.8)
    figure.tight_layout()
    figure.savefig(target, dpi=200)
    plt.close(figure)


In [ ]:
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    objective: OrdinalLoss,
    amp_enabled: bool = False,
) -> Tuple[Dict[str, float], np.ndarray, List[int], List[int], List[str]]:
    model.eval()
    labels: List[int] = []
    predictions: List[int] = []
    names: List[str] = []
    total_loss = 0.0
    total_samples = 0
    elapsed = 0.0
    with torch.inference_mode():
        for images, target, batch_names in loader:
            images = images.to(device, non_blocking=device.type == "cuda")
            target = target.to(device, non_blocking=device.type == "cuda")
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            start = time.perf_counter()
            with torch.autocast(device_type=device.type, enabled=amp_enabled):
                output = model(images)
                losses = objective(output, target)
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            elapsed += time.perf_counter() - start
            prediction = output["logits"].argmax(dim=1)
            total_loss += losses["total"].item() * target.shape[0]
            total_samples += target.shape[0]
            labels.extend(target.cpu().tolist())
            predictions.extend(prediction.cpu().tolist())
            names.extend(batch_names)
    metrics = metrics_from_predictions(labels, predictions)
    metrics["loss"] = total_loss / total_samples
    metrics["inference_ms_per_image"] = 1000.0 * elapsed / total_samples
    return metrics, confusion_matrix(labels, predictions, labels=[0, 1, 2]), labels, predictions, names


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    objective: OrdinalLoss,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    amp_enabled: bool,
) -> Dict[str, float]:
    model.train()
    if isinstance(model, EfficientNetColourFusion) and model.base_frozen:
        # Frozen BatchNorm statistics and classifier dropout must remain in
        # evaluation mode or the supposedly fixed baseline would still drift.
        model.features.eval()
        model.base_classifier.eval()
    total_loss = 0.0
    labels: List[int] = []
    predictions: List[int] = []
    for images, target, _ in loader:
        images = images.to(device, non_blocking=device.type == "cuda")
        target = target.to(device, non_blocking=device.type == "cuda")
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=amp_enabled):
            output = model(images)
            losses = objective(output, target)
        scaler.scale(losses["total"]).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += losses["total"].item() * target.shape[0]
        labels.extend(target.detach().cpu().tolist())
        predictions.extend(output["logits"].argmax(dim=1).detach().cpu().tolist())
    metrics = metrics_from_predictions(labels, predictions)
    metrics["loss"] = total_loss / len(labels)
    return metrics


def parameter_count(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def make_optimizer(model: nn.Module, args: argparse.Namespace) -> torch.optim.Optimizer:
    """Fine-tune ImageNet features conservatively while learning new token heads faster."""
    if isinstance(model, EfficientNetColourFusion) and args.backbone_lr_scale != 1.0:
        backbone = [parameter for parameter in model.features.parameters() if parameter.requires_grad]
        backbone_ids = {id(parameter) for parameter in backbone}
        heads = [
            parameter
            for parameter in model.parameters()
            if parameter.requires_grad and id(parameter) not in backbone_ids
        ]
        if not backbone:
            return torch.optim.AdamW(
                heads, lr=args.learning_rate, weight_decay=args.weight_decay
            )
        return torch.optim.AdamW(
            [
                {"params": backbone, "lr": args.learning_rate * args.backbone_lr_scale},
                {"params": heads, "lr": args.learning_rate},
            ],
            weight_decay=args.weight_decay,
        )
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    return torch.optim.AdamW(trainable, lr=args.learning_rate, weight_decay=args.weight_decay)


def run_training(args: argparse.Namespace, config: ModelConfig, emd_weight: float, score_weight: float) -> Path:
    set_seed(args.seed)
    device = torch.device(args.device)
    train_loader, validation_loader, test_loader = make_loaders(args)
    model = build_model(args.model, config, args.pretrained).to(device)
    if args.base_checkpoint is not None:
        if not isinstance(model, EfficientNetColourFusion):
            raise ValueError("--base-checkpoint is only valid for --model rmof_efficientnet")
        load_frozen_efficientnet_base(model, Path(args.base_checkpoint))
    objective = OrdinalLoss(emd_weight, score_weight, args.label_smoothing)
    optimizer = make_optimizer(model, args)
    amp_enabled = args.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=amp_enabled)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    output_dir = Path(args.output_dir) / args.experiment_name / f"seed_{args.seed}"
    output_dir.mkdir(parents=True, exist_ok=True)
    best_state = copy.deepcopy(model.state_dict())
    best_f1 = -float("inf")
    stale_epochs = 0
    history: List[Dict[str, float]] = []
    if args.base_checkpoint is not None:
        initial_metrics, _, _, _, _ = evaluate(
            model, validation_loader, device, objective, amp_enabled
        )
        best_f1 = initial_metrics["macro_f1"]
        print(f"epoch 000: frozen baseline val macro-F1={best_f1:.4f}")
    for epoch in range(1, args.epochs + 1):
        train_metrics = train_one_epoch(
            model, train_loader, device, objective, optimizer, scaler, amp_enabled
        )
        val_metrics, _, _, _, _ = evaluate(
            model, validation_loader, device, objective, amp_enabled
        )
        scheduler.step()
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "train_macro_f1": train_metrics["macro_f1"],
                "val_loss": val_metrics["loss"],
                "val_macro_f1": val_metrics["macro_f1"],
            }
        )
        print(
            f"epoch {epoch:03d}: train loss={train_metrics['loss']:.4f}, "
            f"val macro-F1={val_metrics['macro_f1']:.4f}"
        )
        if val_metrics["macro_f1"] > best_f1:
            best_f1 = val_metrics["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= args.patience:
                print(f"early stopping after epoch {epoch}")
                break
    model.load_state_dict(best_state)
    test_metrics, cm, labels, predictions, names = evaluate(
        model, test_loader, device, objective, amp_enabled
    )
    test_metrics.update(
        {
            "seed": args.seed,
            "experiment": args.experiment_name,
            "model": args.model,
            "parameters": parameter_count(model),
            "best_validation_macro_f1": best_f1,
            "emd_weight": emd_weight,
            "score_weight": score_weight,
            "base_checkpoint": args.base_checkpoint,
            "training_config": {
                "epochs": args.epochs,
                "batch_size": args.batch_size,
                "augmentation": args.augmentation,
                "train_fraction": args.train_fraction,
                "learning_rate": args.learning_rate,
                "backbone_lr_scale": args.backbone_lr_scale,
                "weight_decay": args.weight_decay,
                "label_smoothing": args.label_smoothing,
                "dropout": config.dropout,
                "amp": amp_enabled,
                "cache_images": args.cache_images,
            },
        }
    )
    with (output_dir / "metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(test_metrics, handle, indent=2)
    pd.DataFrame(history).to_csv(output_dir / "history.csv", index=False)
    pd.DataFrame({"filename": names, "label": labels, "prediction": predictions}).to_csv(
        output_dir / "predictions.csv", index=False
    )
    np.save(output_dir / "confusion.npy", cm)
    save_confusion(cm, output_dir / "confusion_matrix.png", f"{args.experiment_name}, seed {args.seed}")
    torch.save(
        {"state_dict": model.state_dict(), "model_config": asdict(config), "metrics": test_metrics},
        output_dir / "best_model.pt",
    )
    print(json.dumps(test_metrics, indent=2))
    return output_dir


## 5. Results

The following in-notebook table is **Table I**, populated directly from the final values supplied for this project. It is deliberately embedded here so the only Notebook does not depend on a separate CSV. Values with $\pm$ are reported as mean $\pm$ standard deviation; other rows are supplied as point estimates.

**Figure 2** and **Figure 5** are derived directly from the corresponding rows and columns of Table I: no separate ablation or efficiency result tables are maintained. Figure 3 uses the supplied 200-image diagnostic confusion matrices and ordinal-residual frequencies.

**Diagnostic-data note.** The Figure 3 matrices reproduce the supplied diagnostic accuracies and endpoint-error counts. Their per-class F1 and residual frequencies are not sufficient to recompute every Table I metric exactly, so Figure 3 is presented as a descriptive diagnostic view rather than as an independent re-estimation of the main table.

In [ ]:
# Final Table I values supplied for this project. Keeping the data in the notebook
# makes the single-file submission runnable after report artefacts are removed.
FINAL_RESULTS = [
    {"Method": "ResNet18", "Def.-F1": 0.831, "Def.-F1 std": 0.005,
     "Acc.": 0.619, "Acc. std": 0.033, "Macro-F1": 0.588, "Macro-F1 std": 0.031,
     "$F1_{N0}$": 0.777, "$F1_{N0}$ std": 0.020, "$F1_{N75}$": 0.365, "$F1_{N75}$ std": 0.090,
     "$F1_{NFull}$": 0.624, "$F1_{NFull}$ std": 0.088, "Ord.-MAE": 0.411,
     "Ord.-MAE std": 0.028, "Params (M)": 11.178, "Latency (ms/img)": 0.690,
     "Latency std": 0.003, "series": "baseline"},
    {"Method": "DeiT-Tiny", "Def.-F1": 0.764, "Def.-F1 std": 0.088,
     "Acc.": 0.515, "Acc. std": 0.057, "Macro-F1": 0.435, "Macro-F1 std": 0.049,
     "$F1_{N0}$": 0.659, "$F1_{N0}$ std": 0.038, "$F1_{N75}$": 0.136, "$F1_{N75}$ std": 0.139,
     "$F1_{NFull}$": 0.512, "$F1_{NFull}$ std": 0.212, "Ord.-MAE": 0.602,
     "Ord.-MAE std": 0.087, "Params (M)": 5.525, "Latency (ms/img)": 0.795,
     "Latency std": 0.017, "series": "baseline"},
    {"Method": "EfficientNet-B0 (Base)", "Def.-F1": 0.810, "Acc.": 0.585,
     "Macro-F1": 0.552, "$F1_{N0}$": 0.740, "$F1_{N75}$": 0.380,
     "$F1_{NFull}$": 0.610, "Ord.-MAE": 0.460, "Params (M)": 4.011,
     "Latency (ms/img)": 0.555, "series": "ablation"},
    {"Method": "+ multi-scale regions", "Def.-F1": 0.825, "Acc.": 0.612,
     "Macro-F1": 0.585, "$F1_{N0}$": 0.762, "$F1_{N75}$": 0.410,
     "$F1_{NFull}$": 0.645, "Ord.-MAE": 0.420, "Params (M)": 4.069,
     "Latency (ms/img)": 0.565, "series": "ablation"},
    {"Method": "+ explicit colour priors", "Def.-F1": 0.818, "Acc.": 0.605,
     "Macro-F1": 0.578, "$F1_{N0}$": 0.755, "$F1_{N75}$": 0.395,
     "$F1_{NFull}$": 0.632, "Ord.-MAE": 0.435, "Params (M)": 4.073,
     "Latency (ms/img)": 0.994, "series": "ablation"},
    {"Method": "+ learned colour texture", "Def.-F1": 0.845, "Acc.": 0.658,
     "Macro-F1": 0.635, "$F1_{N0}$": 0.790, "$F1_{N75}$": 0.485,
     "$F1_{NFull}$": 0.705, "Ord.-MAE": 0.380, "Params (M)": 4.182,
     "Latency (ms/img)": 1.137, "series": "ablation"},
    {"Method": "+ scalar fusion gate", "Def.-F1": 0.855, "Acc.": 0.672,
     "Macro-F1": 0.650, "$F1_{N0}$": 0.805, "$F1_{N75}$": 0.510,
     "$F1_{NFull}$": 0.725, "Ord.-MAE": 0.365, "Params (M)": 4.214,
     "Latency (ms/img)": 1.137, "series": "ablation"},
    {"Method": "Full RMOF-Net", "Def.-F1": 0.872, "Acc.": 0.715,
     "Macro-F1": 0.698, "$F1_{N0}$": 0.828, "$F1_{N75}$": 0.565,
     "$F1_{NFull}$": 0.768, "Ord.-MAE": 0.315, "Params (M)": 4.314,
     "Latency (ms/img)": 1.143, "series": "ablation"},
]

results = pd.DataFrame(FINAL_RESULTS)
main_metrics = [
    "Def.-F1", "Acc.", "Macro-F1", "$F1_{N0}$", "$F1_{N75}$", "$F1_{NFull}$",
    "Ord.-MAE", "Params (M)", "Latency (ms/img)",
]


def format_result(row: pd.Series, metric: str) -> str:
    """Format a point estimate or the supplied mean ± standard deviation."""
    value = row[metric]
    std_column = f"{metric} std"
    std = row.get(std_column, np.nan)
    return f"{value:.3f} ± {std:.3f}" if pd.notna(std) else f"{value:.3f}"


main_table = pd.DataFrame({"Method": results["Method"]})
for metric in main_metrics:
    main_table[metric] = results.apply(format_result, axis=1, metric=metric)

print("Table I. Main comparison and component analysis")
display(main_table)

# Figure 2: all values are selected from Table I rather than stored separately.
ablation = results.loc[results["series"].eq("ablation")].reset_index(drop=True)
short_names = [
    "Base", "Multi-scale", "Colour priors", "Colour texture", "Scalar gate", "Full RMOF-Net",
]
bar_colours = ["#0072B2"] + ["#56B4E9"] * 4 + ["#D55E00"]
fig2, axes = plt.subplots(1, 3, figsize=(14, 4.1), constrained_layout=True)
for axis, metric, title in zip(
    axes,
    ["Macro-F1", "$F1_{N75}$", "Ord.-MAE"],
    ["Macro-F1 (higher is better)", "$F1_{N75}$ (higher is better)", "Ordinal MAE (lower is better)"],
):
    positions = np.arange(len(ablation))
    axis.barh(positions, ablation[metric], color=bar_colours, edgecolor="white")
    axis.set_yticks(positions, short_names)
    axis.invert_yaxis()
    axis.set_xlabel(metric)
    axis.set_title(title, fontsize=10)
    axis.grid(axis="x", alpha=0.22)
    axis.set_axisbelow(True)
    if metric != "Ord.-MAE":
        axis.set_xlim(0, 0.8)
    else:
        axis.set_xlim(0, 0.5)
plt.show()

# Figure 3: supplied 200-image diagnostic data, kept separate from Table I.
diagnostic_matrices = {
    "EfficientNet-B0 (Base)": np.array([[42, 16, 12], [10, 38, 12], [12, 21, 37]]),
    "Full RMOF-Net": np.array([[47, 21, 3], [8, 42, 10], [2, 14, 54]]),
}
residual_counts = pd.DataFrame(
    {
        "Residual": [-2, -1, 0, 1, 2],
        "EfficientNet-B0 (Base)": [12, 33, 117, 26, 12],
        "Full RMOF-Net": [2, 22, 143, 28, 5],
    }
)

fig3, axes = plt.subplots(1, 3, figsize=(13.2, 3.9), constrained_layout=True)
for axis, (method, matrix) in zip(axes[:2], diagnostic_matrices.items()):
    image = axis.imshow(matrix, cmap="Blues", vmin=0, vmax=55)
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            axis.text(column, row, str(matrix[row, column]), ha="center", va="center", fontsize=10)
    axis.set_xticks(range(3), CLASS_NAMES)
    axis.set_yticks(range(3), CLASS_NAMES)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")
    accuracy = matrix.trace() / matrix.sum()
    endpoints = int(matrix[0, 2] + matrix[2, 0])
    axis.set_title(f"{method}\nAcc.={accuracy:.1%}, endpoint errors={endpoints}", fontsize=10)
fig3.colorbar(image, ax=axes[:2], fraction=0.046, pad=0.03, label="Images")

positions = np.arange(len(residual_counts))
width = 0.36
axes[2].bar(positions - width / 2, residual_counts["EfficientNet-B0 (Base)"], width,
            color="#0072B2", label="EfficientNet-B0")
axes[2].bar(positions + width / 2, residual_counts["Full RMOF-Net"], width,
            color="#D55E00", label="Full RMOF-Net")
axes[2].set_xticks(positions, ["-2", "-1", "0", "+1", "+2"])
axes[2].set_xlabel(r"Ordinal residual $\hat{s}-s$")
axes[2].set_ylabel("Images")
axes[2].set_title("Ordinal residual distribution", fontsize=10)
axes[2].legend(frameon=False, fontsize=8)
axes[2].grid(axis="y", alpha=0.22)
axes[2].set_axisbelow(True)
plt.show()

# Figure 5: both scatter plots reuse Table I directly.
fig5, axes = plt.subplots(1, 2, figsize=(11.5, 4.0), constrained_layout=True)
for axis, x_metric, x_label in zip(
    axes,
    ["Latency (ms/img)", "Params (M)"],
    ["Latency (ms/image, lower is better)", "Trainable parameters (M, lower is better)"],
):
    for _, row in results.iterrows():
        is_full = row["Method"] == "Full RMOF-Net"
        is_base = row["Method"] == "EfficientNet-B0 (Base)"
        colour = "#D55E00" if is_full else ("#0072B2" if is_base else "#666666")
        marker = "*" if is_full else "o"
        size = 130 if is_full else 55
        axis.scatter(row[x_metric], row["Macro-F1"], s=size, marker=marker, color=colour, zorder=3)
        label = row["Method"].replace("EfficientNet-B0 (Base)", "EfficientNet-B0")
        axis.annotate(label, (row[x_metric], row["Macro-F1"]), xytext=(4, 4),
                      textcoords="offset points", fontsize=7)
    axis.set_xlabel(x_label)
    axis.set_ylabel("Macro-F1 (higher is better)")
    axis.set_ylim(0.38, 0.75)
    axis.grid(alpha=0.22)
    axis.set_axisbelow(True)
plt.show()

## 6. Discussion

### Findings

Full RMOF-Net has the highest supplied Table I score for every effectiveness metric: Macro-F1 rises from 0.552 for EfficientNet-B0 (Base) to 0.698, $F1_{N75}$ rises from 0.380 to 0.565, and ordinal MAE falls from 0.460 to 0.315. The component rows show the strongest improvement after learned colour texture and then scalar fusion. Relative to the base, Full RMOF-Net uses 0.303M more trainable parameters and adds 0.588 ms/image latency, so the accuracy gain has a clear inference-cost trade-off.

Figure 3 provides the supplied error-analysis view. The diagnostic confusion matrices show 24 endpoint errors for EfficientNet-B0 (Base) and 5 for Full RMOF-Net. The residual plot also concentrates more observations at zero for Full RMOF-Net. These diagnostic counts are descriptive; as stated above, they should not be used to recompute the seed-level metrics in Table I.

### Strengths

The system combines global plant context with regional colour evidence, reports Macro-F1 and class-specific F1 rather than accuracy alone, and measures ordinal severity and deployment cost. The largest gain occurs on the difficult middle class ($N75$), which is a useful result because a method that improves only the endpoints would be less convincing for an ordered treatment task.

### Limitations and Future Work

Only ResNet18 and DeiT-Tiny include supplied standard deviations; the EfficientNet-B0 series is shown as point estimates. A final matched multi-seed protocol, shared tuning budget, confidence intervals, peak-memory measurement, and prediction files are still needed before making statistical claims. Treatment labels proxy nitrogen application rather than direct tissue nitrogen, and the dataset has no external-farm validation. Future work should evaluate across farms, cameras, growth stages, weather, and genotypes, then relate predictions to tissue measurements and yield.

## 7. Reproducibility and Optional Training

The notebook contains the model, baseline, ablation, loss, evaluation, and training implementations in this file. It reads only the bundled images and split manifest from `EfficientN/`; it does not import project source files or depend on report-side CSV, figure, or PDF files. The final code cell runs a CPU smoke check automatically. Set `RUN_TRAINING = True` only when intentionally running the in-notebook baseline and ablation suite; training outputs are written below `EfficientN/notebook_outputs/`.

In [ ]:
def smoke_check(device: torch.device) -> None:
    """Check baseline equivalence, then run the custom forward/loss/backward path."""
    config = preset_config("ordinal_supervision")
    model = EfficientNetColourFusion(config, pretrained=False).to(device)
    images = torch.randn(2, 3, 128, 128, device=device)
    target = torch.tensor([0, 2], device=device)
    model.eval()
    with torch.inference_mode():
        initial_output = model(images)
    if not torch.equal(initial_output["logits"], initial_output["base_logits"]):
        raise RuntimeError("Zero-initialized auxiliary path changed the baseline logits")
    if torch.count_nonzero(initial_output["aux_logits"]):
        raise RuntimeError("Auxiliary logits must be exactly zero at initialization")
    model.train()
    output = model(images)
    assert output["logits"].shape == (2, 3)
    assert output["score"] is not None and output["score"].shape == (2, 1)
    losses = OrdinalLoss(0.25, 0.25)(output, target)
    losses["total"].backward()
    if not any(parameter.grad is not None for parameter in model.parameters()):
        raise RuntimeError("Backward pass did not produce gradients")
    if model.aux_classifier[-1].weight.grad is None:
        raise RuntimeError("Auxiliary classifier did not receive gradients")
    print(
        "smoke check passed: exact baseline initialization, logits=(2,3), "
        "score=(2,1), forward/backward/loss all valid"
    )


In [ ]:
# Everything below uses the implementations defined in this notebook.
# Training remains opt-in because the complete suite is compute-intensive.
RUN_TRAINING = False

NOTEBOOK_EXPERIMENTS = [
    ("resnet18", "resnet18", "cnn_baseline", 0.0, 0.0),
    ("deit_tiny", "deit_tiny", "cnn_baseline", 0.0, 0.0),
    ("efficientnet_b0", "efficientnet_b0", "cnn_baseline", 0.0, 0.0),
    ("multiscale_regions", "rmof_efficientnet", "regions", 0.0, 0.0),
    ("colour_stats", "rmof_efficientnet", "color_stats", 0.0, 0.0),
    ("colour_texture", "rmof_efficientnet", "color_texture", 0.0, 0.0),
    ("scalar_gate", "rmof_efficientnet", "fusion_gate", 0.0, 0.0),
    ("full_rmof", "rmof_efficientnet", "ordinal_supervision", 0.25, 0.25),
]

def notebook_args(experiment: str, model: str, seed: int, epochs: int) -> SimpleNamespace:
    return SimpleNamespace(
        csv=str(EXPERIMENT_ROOT / "split_40_10_50.csv"),
        data_root=str(EXPERIMENT_ROOT),
        output_dir=str(EXPERIMENT_ROOT / "notebook_outputs"),
        experiment_name=experiment,
        model=model,
        pretrained=True,
        seed=seed,
        epochs=epochs,
        patience=10,
        batch_size=32,
        num_workers=2,
        image_size=224,
        augmentation="mild",
        train_fraction=1.0,
        learning_rate=1e-3 if model == "rmof_efficientnet" else 3e-4,
        weight_decay=1e-2,
        backbone_lr_scale=0.3,
        label_smoothing=0.1,
        cache_images=True,
        amp=True,
        device="cuda" if torch.cuda.is_available() else "cpu",
        base_checkpoint=None,
    )

def run_notebook_suite(seeds: Sequence[int] = (42, 52, 62), epochs: int = 10) -> pd.DataFrame:
    records = []
    for experiment, model_name, preset, emd_weight, score_weight in NOTEBOOK_EXPERIMENTS:
        for seed in seeds:
            args = notebook_args(experiment, model_name, seed, epochs)
            config = replace(preset_config(preset), dropout=0.4)
            output_dir = run_training(args, config, emd_weight, score_weight)
            with (output_dir / "metrics.json").open(encoding="utf-8") as handle:
                metrics = json.load(handle)
            records.append({"Method": experiment, **metrics})
    return pd.DataFrame(records)

smoke_check(torch.device("cpu"))
if RUN_TRAINING:
    notebook_results = run_notebook_suite(seeds=(42, 52, 62), epochs=10)
    display(notebook_results)
else:
    print("Notebook is self-contained. Set RUN_TRAINING = True to run the baseline and ablation suite.")
